# Performance Comparison — MinerU2.5 Deployment Options

Reads from the shared performance results table and visualises key metrics across deployment options:
- **End-to-end latency** per document (bar chart)
- **Cold start time** (average per option)
- **Throughput** in pages/min (average per option)
- **Trade-off summary** table with recommendations

### Cluster Requirements

| Setting | Value |
|---------|-------|
| Instance | Serverless |
| Libraries | None — installed via `%pip` below |

### Prerequisites
- Run `vllm_batch/tests` and `vllm_rt/tests` first to populate the perf table

### Install Dependencies

`plotly` is used for interactive bar charts. `pyyaml` reads the shared config file.

In [ ]:
%pip install plotly pyyaml

### Configuration

In [ ]:
import yaml, os

# Resolve project root
if "__file__" in dir():
    _root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
else:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _root = "/Workspace" + os.path.dirname(os.path.dirname(_nb))

cfg        = yaml.safe_load(open(f"{_root}/config.yaml"))
CATALOG    = cfg["catalog"]
SCHEMA     = cfg["schema"]
PERF_TABLE = f"{CATALOG}.{SCHEMA}.{cfg['perf_table']}"
print(f"Perf table: {PERF_TABLE}")

### Load Performance Data

Reads the perf table, filters to `vLLM_Batch` and `vLLM_RT` options, and computes throughput (pages per minute) as a derived column.

In [ ]:
from pyspark.sql import functions as F

# Load perf data, filter to active options, compute throughput
df = (spark.table(PERF_TABLE)
      .filter(F.col("option").isin("vLLM_Batch", "vLLM_RT"))
      .withColumn("pages_per_min", F.col("pages") / (F.col("latency_s") / 60))
      .orderBy("option", "tc_id"))

display(df)

## Latency by Option and Test Case

Grouped bar chart showing end-to-end latency per document. For **vLLM_Batch**, this includes cold start (cluster startup + model loading). For **vLLM_RT**, this is pure inference time since the model is always hot.

In [ ]:
import plotly.express as px

# Convert to Pandas for Plotly
pdf = df.toPandas()

fig = px.bar(
    pdf, x="tc_id", y="latency_s", color="option", barmode="group",
    title="End-to-End Latency per Document (seconds)",
    labels={"latency_s": "Latency (s)", "tc_id": "Test Case", "option": "Option"},
)
fig.show()

## Cold Start Time by Option

Average cold start across test cases. **vLLM_Batch** includes cluster startup + model loading (~810s). **vLLM_RT** is always 0 because the model is pre-loaded by the continuous job.

In [ ]:
# Average cold start per option
cold_start = (df.groupBy("option")
               .agg(F.avg("cold_start_s").alias("avg_cold_start_s"))
               .orderBy("option"))
display(cold_start)

## Throughput (pages/min) by Option

Average throughput in pages per minute. Higher is better. **vLLM_RT** typically achieves ~60 pages/min (hot model), while **vLLM_Batch** is dominated by cold start overhead.

In [ ]:
# Average throughput per option
throughput = (df.groupBy("option")
               .agg(F.avg("pages_per_min").alias("avg_pages_per_min"))
               .orderBy("option"))
display(throughput)

## Trade-off Summary

| Dimension | vLLM_Batch | vLLM_RT |
|-----------|------------|---------|
| **Pattern** | Triggered batch job | Continuous job + driver proxy |
| **Inference** | vLLM (offline) | vLLM (online, OpenAI-compatible API) |
| **Latency** | High (~810s incl. cold start) | Low (~1s/page, model always hot) |
| **Cold Start** | ~810s (cluster + model load) | None (always running) |
| **Scalability** | Single job per cluster | Single GPU, not scalable |
| **Cost** | Low (pay-per-use, idle = $0) | High (always-on GPU) |
| **Best For** | Scheduled batch workloads | Demos, prototyping, interactive use |

### Recommendations

- **Use vLLM_Batch** for scheduled/recurring workloads where latency tolerance is high and cost efficiency matters. The cluster spins up only when there are pending requests.
- **Use vLLM_RT** for demos, prototyping, or interactive applications that need sub-second per-page latency. Be mindful of the always-on GPU cost — pause the job when not in use.